
# Classification binaire des compétences IA / non-IA — Deep Learning TextCNN

Objectif : entraîner un modèle TextCNN réellement from scratch, avec vocabulaire appris uniquement sur le train, mêmes splits que le notebook Machine Learning, même seed et mêmes métriques finales.



```bash
pip install pandas numpy scikit-learn matplotlib openpyxl joblib torch
```


In [ ]:

from __future__ import annotations

import json
import random
import sys
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

ROOT = None
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / '.git').exists() and (candidate / 'data' / 'raw').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Impossible de localiser la racine du dépôt.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.ia_non_ia_shared import (  # noqa: E402
    SEED,
    audit_dataset,
    build_threshold_table,
    choose_threshold_from_validation,
    enrich_with_ids,
    evaluate_with_threshold,
    find_repo_root,
    load_or_create_splits,
    model_size_bytes,
    package_versions,
    plot_class_distribution,
    plot_confusion_matrix,
    plot_length_distribution,
    plot_probability_distribution,
    plot_roc_pr_calibration,
    plot_threshold_metrics,
    read_dataset,
    report_to_flat_row,
    save_json,
    split_frames,
    summarize_split_sizes,
    threshold_grid,
)
from deepforma.training.binary_ai_textcnn import (  # noqa: E402
    PAD_TOKEN,
    UNK_TOKEN,
    BinaryAIDataset,
    TextCNN,
    build_vocabulary,
    encode_text,
    tokenize,
)

ROOT = find_repo_root(ROOT)
DATASET_PATH = ROOT / 'data' / 'raw' / 'dataset_competences_IA_annotees.xlsx'
ARTIFACT_DIR = ROOT / 'artifacts' / 'classification_ia_non_ia_textcnn'
COMMON_SPLIT_PATH = ROOT / 'artifacts' / 'classification_ia_non_ia_common' / 'splits.csv'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
COMMON_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = None
LABEL_COLUMN = None
THRESHOLD_GRID = threshold_grid(0.05, 0.95, 0.01)

VOCAB_SIZE = 8000
MAX_LENGTH = 32
EMBEDDING_DIM = 128
KERNEL_SIZES = (2, 3, 4, 5)
NUM_FILTERS = 96
HIDDEN_DIM = 128
DROPOUT = 0.4
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 20
PATIENCE = 5
GRAD_CLIP = 1.0


In [ ]:

def normalize_name(value: str) -> str:
    normalized = unicodedata.normalize('NFKD', str(value))
    normalized = ''.join(ch for ch in normalized if not unicodedata.combining(ch))
    return normalized.lower().strip()


def detect_columns(frame: pd.DataFrame) -> tuple[str, str]:
    normalized = {normalize_name(column): column for column in frame.columns}
    text_candidates = ['competence', 'compétence', 'texte', 'text']
    label_candidates = ['ia', 'label', 'cible', 'target', 'classe']
    text_column = next((normalized[name] for name in text_candidates if name in normalized), None)
    label_column = next((normalized[name] for name in label_candidates if name in normalized), None)
    if text_column is None or label_column is None:
        raise ValueError(f'Colonnes attendues introuvables. Colonnes disponibles: {list(frame.columns)}')
    return text_column, label_column


def to_binary_frame(frame: pd.DataFrame, text_column: str, label_column: str) -> pd.DataFrame:
    output = frame.copy()
    output['text'] = output[text_column].astype(str)
    output['is_ai'] = output[label_column].astype(int)
    return output[['source_index', 'record_id', text_column, label_column, 'split', 'text', 'is_ai']].copy()


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sequence_length_stats(texts: pd.Series) -> pd.DataFrame:
    lengths = texts.astype(str).map(lambda value: len(tokenize(value)))
    return pd.DataFrame({
        'statistique': ['min', 'moyenne', 'médiane', 'max', 'p10', 'p25', 'p75', 'p90'],
        'valeur': [
            int(lengths.min()), float(lengths.mean()), float(lengths.median()), int(lengths.max()),
            float(lengths.quantile(0.10)), float(lengths.quantile(0.25)), float(lengths.quantile(0.75)), float(lengths.quantile(0.90)),
        ],
    })


def unknown_token_rate(texts: pd.Series, vocab: dict[str, int], max_length: int) -> float:
    unknown = 0
    total = 0
    for text in texts.astype(str):
        tokens = tokenize(text)[:max_length]
        total += len(tokens)
        unknown += sum(1 for token in tokens if token not in vocab)
    return float(unknown / total) if total else 0.0


def build_loaders(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, test_frame: pd.DataFrame, vocab: dict[str, int]):
    train_dataset = BinaryAIDataset(train_frame, vocab, max_length=MAX_LENGTH)
    valid_dataset = BinaryAIDataset(valid_frame, vocab, max_length=MAX_LENGTH)
    test_dataset = BinaryAIDataset(test_frame, vocab, max_length=MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, valid_loader, test_loader


def evaluate_scores(model: TextCNN, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    scores: list[float] = []
    labels: list[int] = []
    with torch.no_grad():
        for input_ids, targets in loader:
            input_ids = input_ids.to(device)
            logits = model(input_ids)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            scores.extend(float(value) for value in probs.tolist())
            labels.extend(int(value) for value in targets.tolist())
    return np.asarray(labels, dtype=int), np.asarray(scores, dtype=float)


def train_textcnn(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, test_frame: pd.DataFrame):
    seed_everything(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab = build_vocabulary(train_frame['text'].tolist(), max_size=VOCAB_SIZE)
    train_loader, valid_loader, test_loader = build_loaders(train_frame, valid_frame, test_frame, vocab)

    model = TextCNN(
        vocab_size=len(vocab),
        embedding_dim=EMBEDDING_DIM,
        num_filters=NUM_FILTERS,
        kernel_sizes=KERNEL_SIZES,
        dense_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(device)

    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    y_train = train_frame['is_ai'].to_numpy(dtype=int)
    n_pos = int(y_train.sum())
    n_neg = int(len(y_train) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    history: list[dict[str, float]] = []
    best_state = None
    best_pr_auc = -1.0
    epochs_without_improvement = 0
    start_training = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        batch_losses: list[float] = []
        for input_ids, targets in train_loader:
            input_ids = input_ids.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids)
            loss = criterion(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_loss = float(np.mean(batch_losses)) if batch_losses else 0.0
        valid_labels, valid_scores = evaluate_scores(model, valid_loader, device)
        threshold_metrics = evaluate_with_threshold(
            valid_labels,
            valid_scores,
            threshold=0.5,
            model_name='textcnn_epoch_validation',
            inference_time_seconds=None,
            model_size_bytes=None,
        )[0]
        val_pr_auc = float(threshold_metrics['pr_auc']) if threshold_metrics['pr_auc'] is not None else 0.0
        val_roc_auc = float(threshold_metrics['roc_auc']) if threshold_metrics['roc_auc'] is not None else 0.0
        val_f1_05 = float(threshold_metrics['f1_ia']) if threshold_metrics['f1_ia'] is not None else 0.0
        current_lr = float(optimizer.param_groups[0]['lr'])

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'validation_pr_auc': val_pr_auc,
            'validation_roc_auc': val_roc_auc,
            'validation_f1_at_0_5': val_f1_05,
            'learning_rate': current_lr,
        })

        scheduler.step(val_pr_auc)
        if val_pr_auc > best_pr_auc + 1e-8:
            best_pr_auc = val_pr_auc
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Arrêt anticipé à l'époque {epoch}.")
                break

        print(
            f"Epoch {epoch:02d} | loss={train_loss:.4f} | PR-AUC={val_pr_auc:.4f} | ROC-AUC={val_roc_auc:.4f} | F1@0.5={val_f1_05:.4f} | lr={current_lr:.2e}"
        )

    training_time_seconds = time.perf_counter() - start_training
    if best_state is not None:
        model.load_state_dict(best_state)

    start = time.perf_counter()
    train_labels, train_scores = evaluate_scores(model, train_loader, device)
    train_inference_time = time.perf_counter() - start
    start = time.perf_counter()
    valid_labels, valid_scores = evaluate_scores(model, valid_loader, device)
    valid_inference_time = time.perf_counter() - start
    start = time.perf_counter()
    test_labels, test_scores = evaluate_scores(model, test_loader, device)
    test_inference_time = time.perf_counter() - start

    threshold_table_validation = build_threshold_table(valid_labels, valid_scores, thresholds=THRESHOLD_GRID)
    threshold, best_threshold_row = choose_threshold_from_validation(threshold_table_validation)

    validation_metrics, validation_report = evaluate_with_threshold(
        valid_labels,
        valid_scores,
        threshold=threshold,
        model_name='textcnn',
        inference_time_seconds=test_inference_time,
        model_size_bytes=None,
    )
    test_metrics, test_report = evaluate_with_threshold(
        test_labels,
        test_scores,
        threshold=threshold,
        model_name='textcnn',
        inference_time_seconds=None,
        model_size_bytes=None,
    )

    model_path = ARTIFACT_DIR / 'textcnn_ia_non_ia.pt'
    config_path = ARTIFACT_DIR / 'config.json'
    vocab_path = ARTIFACT_DIR / 'vocabulary.json'
    history_path = ARTIFACT_DIR / 'training_history.csv'
    threshold_path = ARTIFACT_DIR / 'threshold.json'
    metadata_path = ARTIFACT_DIR / 'metadata.json'

    payload = {
        'model_state_dict': model.state_dict(),
        'threshold': float(threshold),
        'vocab': vocab,
        'config': {
            'vocab_size': VOCAB_SIZE,
            'max_length': MAX_LENGTH,
            'embedding_dim': EMBEDDING_DIM,
            'kernel_sizes': list(KERNEL_SIZES),
            'num_filters': NUM_FILTERS,
            'hidden_dim': HIDDEN_DIM,
            'dropout': DROPOUT,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS,
            'patience': PATIENCE,
            'grad_clip': GRAD_CLIP,
            'seed': SEED,
        },
        'trainable_parameters': int(trainable_parameters),
    }
    torch.save(payload, model_path)
    save_json(vocab_path, vocab)
    save_json(config_path, payload['config'])
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False, encoding='utf-8')

    model_size_bytes_value = model_size_bytes(model_path)
    validation_metrics['training_time_seconds'] = training_time_seconds
    validation_metrics['model_size_bytes'] = model_size_bytes_value
    validation_metrics['model_size_mb'] = model_size_bytes_value / (1024 * 1024)
    test_metrics['training_time_seconds'] = training_time_seconds
    test_metrics['model_size_bytes'] = model_size_bytes_value
    test_metrics['model_size_mb'] = model_size_bytes_value / (1024 * 1024)

    save_json(threshold_path, {
        'model_name': 'textcnn',
        'threshold': float(threshold),
        'selection_rule': [
            'meilleur F1 macro',
            'meilleur F1 IA',
            'meilleur rappel IA',
            "puis seuil le plus proche de 0.5 en cas d'égalité"
        ],
        'threshold_grid': [float(x) for x in THRESHOLD_GRID.tolist()],
        'best_threshold_row': best_threshold_row.to_dict(),
        'validation_threshold_table_rows': int(len(threshold_table_validation)),
        'date': pd.Timestamp.utcnow().isoformat(),
    })

    save_json(ARTIFACT_DIR / 'metrics_validation.json', validation_metrics)
    save_json(ARTIFACT_DIR / 'metrics_test.json', test_metrics)
    history_df.to_csv(history_path, index=False, encoding='utf-8')

    errors_test = build_error_analysis(test_frame, test_scores, threshold)
    errors_test.to_csv(ARTIFACT_DIR / 'error_analysis.csv', index=False, encoding='utf-8')

    metadata = {
        'model_type': 'TextCNN',
        'pretrained_model': False,
        'pretrained_embeddings': False,
        'random_initialization': True,
        'seed': SEED,
        'vocab_size': len(vocab),
        'embedding_dim': EMBEDDING_DIM,
        'kernel_sizes': list(KERNEL_SIZES),
        'num_filters': NUM_FILTERS,
        'hidden_dim': HIDDEN_DIM,
        'dropout': DROPOUT,
        'max_length': MAX_LENGTH,
        'trainable_parameters': int(trainable_parameters),
        'colonnes_utilisees': {'texte': text_column, 'cible': label_column},
        'nombre_exemples_par_split': {'train': len(train_frame), 'validation': len(valid_frame), 'test': len(test_frame)},
        'training_time_seconds': float(training_time_seconds),
        'threshold': float(threshold),
        'model_size_bytes': int(model_size_bytes_value),
        'versions_des_bibliotheques': package_versions(['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'openpyxl', 'joblib', 'torch']),
        'date_entrainement': pd.Timestamp.utcnow().isoformat(),
    }
    save_json(metadata_path, metadata)

    return {
        'model': model,
        'device': device,
        'vocab': vocab,
        'history': history_df,
        'threshold': float(threshold),
        'validation_metrics': validation_metrics,
        'validation_report': validation_report,
        'test_metrics': test_metrics,
        'test_report': test_report,
        'trainable_parameters': int(trainable_parameters),
        'training_time_seconds': float(training_time_seconds),
        'model_size_bytes': int(model_size_bytes_value),
        'errors_test': errors_test,
        'payload': payload,
    }


def build_error_analysis(frame: pd.DataFrame, scores: np.ndarray, threshold: float) -> pd.DataFrame:
    predictions = (scores >= threshold).astype(int)
    errors = frame.loc[predictions != frame['is_ai'].to_numpy(dtype=int)].copy()
    if errors.empty:
        return pd.DataFrame(columns=['source_index', 'record_id', 'split', 'text', 'true_label', 'predicted_label', 'probability_ia', 'threshold', 'error_type'])
    errors['true_label'] = errors['is_ai'].map({0: 'non-IA', 1: 'IA'})
    errors['predicted_label'] = pd.Series(predictions, index=frame.index).loc[errors.index].map({0: 'non-IA', 1: 'IA'})
    score_series = pd.Series(scores, index=frame.index)
    errors['probability_ia'] = score_series.loc[errors.index].to_numpy()
    errors['threshold'] = threshold
    errors['error_type'] = np.where(errors['is_ai'] == 1, 'faux négatif', 'faux positif')
    return errors[['source_index', 'record_id', 'split', 'text', 'true_label', 'predicted_label', 'probability_ia', 'threshold', 'error_type']].sort_values('probability_ia', ascending=False).reset_index(drop=True)


def predict_competence_textcnn(text: str, *, model_dir: Path | None = None) -> dict[str, object]:
    model_dir = model_dir or ARTIFACT_DIR
    payload = torch.load(model_dir / 'textcnn_ia_non_ia.pt', map_location='cpu')
    config = payload['config']
    vocab = payload['vocab']
    model = TextCNN(
        vocab_size=len(vocab),
        embedding_dim=config['embedding_dim'],
        num_filters=config['num_filters'],
        kernel_sizes=config['kernel_sizes'],
        dense_dim=config['hidden_dim'],
        dropout=config['dropout'],
    )
    model.load_state_dict(payload['model_state_dict'])
    model.eval()
    encoded = torch.tensor([encode_text(str(text), vocab, max_length=config['max_length'])], dtype=torch.long)
    with torch.no_grad():
        probability = float(torch.sigmoid(model(encoded))[0].item())
    threshold = float(payload.get('threshold', 0.5))
    prediction = 'IA' if probability >= threshold else 'non-IA'
    return {
        'competence': str(text),
        'probability_ia': probability,
        'threshold': threshold,
        'prediction': prediction,
    }


In [ ]:

# Chargement du dataset, audit et visualisations
raw = read_dataset(DATASET_PATH)
text_column, label_column = detect_columns(raw)
TEXT_COLUMN, LABEL_COLUMN = text_column, label_column

audit = audit_dataset(raw, text_column=text_column, label_column=label_column)
print(f"Feuille source: Dataset")
print(f"Colonnes détectées: texte='{text_column}', cible='{label_column}'")
print(f"Nombre de lignes: {audit['n_rows']}")
print(f"Distribution des classes: {audit['label_distribution']}")
print(f"Valeurs manquantes: {audit['missing_by_column']}")
print(f"Doublons exacts: {audit['exact_duplicates']}")
print(f"Contradictions d'étiquettes: {audit['contradictory_labels_by_text']}")
print(f"Textes très courts: {audit['short_text_counts']}")

display(pd.DataFrame({
    'métrique': [
        'Nombre de lignes', 'Nombre de colonnes', 'Distribution classe 0', 'Distribution classe 1',
        'Doublons exacts', "Contradictions d'étiquettes", 'Textes < 5 caractères', 'Textes < 3 mots'
    ],
    'valeur': [
        audit['n_rows'], audit['n_columns'], audit['label_distribution'].get(0, 0), audit['label_distribution'].get(1, 0),
        audit['exact_duplicates'], audit['contradictory_labels_by_text'], audit['short_text_counts']['len_chars_lt_5'], audit['short_text_counts']['len_words_lt_3']
    ]
}))

display(pd.DataFrame({'colonne': list(audit['missing_by_column'].keys()), 'valeurs_manquantes': list(audit['missing_by_column'].values())}))

fig, ax = plt.subplots(figsize=(5, 4))
counts = pd.Series({0: audit['label_distribution'].get(0, 0), 1: audit['label_distribution'].get(1, 0)})
counts.index = ['non-IA', 'IA']
counts.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518'])
ax.set_title('Distribution des classes')
ax.set_ylabel("Nombre d'exemples")
ax.set_xlabel('Classe')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
text_lengths = raw[text_column].astype(str).str.strip().str.len()
ax.hist(text_lengths, bins=30, color='#4c78a8', alpha=0.85)
ax.set_title('Distribution des longueurs des compétences')
ax.set_xlabel('Nombre de caractères')
ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.show()


In [ ]:

# Split stratifié commun et préparation des données pour TextCNN
base = enrich_with_ids(raw)
base = load_or_create_splits(base, label_column=label_column, seed=SEED, split_path=COMMON_SPLIT_PATH)
frames = split_frames(base)

df_train = to_binary_frame(frames['train'], text_column=text_column, label_column=label_column)
df_valid = to_binary_frame(frames['validation'], text_column=text_column, label_column=label_column)
df_test = to_binary_frame(frames['test'], text_column=text_column, label_column=label_column)

display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n_samples': [len(df_train), len(df_valid), len(df_test)],
    'n_ia': [int(df_train['is_ai'].sum()), int(df_valid['is_ai'].sum()), int(df_test['is_ai'].sum())],
    'n_non_ia': [int((df_train['is_ai'] == 0).sum()), int((df_valid['is_ai'] == 0).sum()), int((df_test['is_ai'] == 0).sum())],
}))

sequence_stats = sequence_length_stats(df_train['text'])
display(sequence_stats)

fig, ax = plt.subplots(figsize=(6, 4))
train_token_lengths = df_train['text'].astype(str).map(lambda value: len(tokenize(value)))
ax.hist(train_token_lengths, bins=20, color='#72b7b2', alpha=0.85)
ax.set_title('Longueur des séquences en mots — train')
ax.set_xlabel('Nombre de tokens')
ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.show()

assert set(df_train['is_ai'].unique()) <= {0, 1}
assert set(df_valid['is_ai'].unique()) <= {0, 1}
assert set(df_test['is_ai'].unique()) <= {0, 1}
assert df_train['is_ai'].nunique() == 2 and df_valid['is_ai'].nunique() == 2 and df_test['is_ai'].nunique() == 2


In [ ]:

# Vocabulaire appris uniquement sur le train et contrôle des tokens inconnus
seed_everything(SEED)
vocab = build_vocabulary(df_train['text'].tolist(), max_size=VOCAB_SIZE)
print('Taille du vocabulaire (avec PAD/UNK):', len(vocab))
print('Token PAD:', PAD_TOKEN, 'Token UNK:', UNK_TOKEN)
print('Taux de tokens inconnus validation:', unknown_token_rate(df_valid['text'], vocab, MAX_LENGTH))
print('Taux de tokens inconnus test:', unknown_token_rate(df_test['text'], vocab, MAX_LENGTH))

display(pd.DataFrame({'statistique': ['taille_vocabulaire', 'taux_UNK_validation', 'taux_UNK_test', 'longueur_max'], 'valeur': [len(vocab), unknown_token_rate(df_valid['text'], vocab, MAX_LENGTH), unknown_token_rate(df_test['text'], vocab, MAX_LENGTH), MAX_LENGTH]}))


In [ ]:

# Entraînement du TextCNN from scratch avec early stopping sur PR-AUC validation
results = train_textcnn(df_train, df_valid, df_test)

history = results['history']
display(history)

display(
    pd.DataFrame(
        [
            report_to_flat_row('textcnn', results['validation_metrics']),
            report_to_flat_row('textcnn', results['test_metrics']),
        ]
    ).assign(split=['validation', 'test'])
)
print("Seuil retenu:", results['threshold'])
print("Paramètres entraînables:", results['trainable_parameters'])
print("Appareil utilisé:", results['device'])
print("Temps d'entraînement (s):", results['training_time_seconds'])
print("Taille du modèle (octets):", results['model_size_bytes'])


In [ ]:

# Courbes d'entraînement, métriques finales, calibration et erreurs
history_df = results['history']
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
ax.set_title("Loss d'entraînement")
ax.set_xlabel('Époque')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df['epoch'], history_df['validation_pr_auc'], label='validation_pr_auc')
ax.plot(history_df['epoch'], history_df['validation_roc_auc'], label='validation_roc_auc')
ax.plot(history_df['epoch'], history_df['validation_f1_at_0_5'], label='validation_f1@0.5')
ax.set_title('Métriques validation par époque')
ax.set_xlabel('Époque')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df['epoch'], history_df['learning_rate'], label='learning_rate')
ax.set_title('Learning rate par époque')
ax.set_xlabel('Époque')
ax.set_ylabel('LR')
ax.legend()
plt.tight_layout()
plt.show()

train_loader, valid_loader, test_loader = build_loaders(df_train, df_valid, df_test, results['vocab'])
valid_labels, valid_scores = evaluate_scores(results['model'], valid_loader, results['device'])
test_labels, test_scores = evaluate_scores(results['model'], test_loader, results['device'])
threshold_table_validation = build_threshold_table(valid_labels, valid_scores, thresholds=THRESHOLD_GRID)
plot_threshold_metrics(threshold_table_validation, title='Scores selon le seuil — validation')
plot_confusion_matrix(results['validation_report'], title='Matrice de confusion — validation')
plot_confusion_matrix(results['test_report'], title='Matrice de confusion — test')
plot_roc_pr_calibration(results['validation_report'], prefix='textcnn_validation', output_dir=ARTIFACT_DIR)
plot_roc_pr_calibration(results['test_report'], prefix='textcnn_test', output_dir=ARTIFACT_DIR)
plot_probability_distribution(valid_scores, title='Distribution des probabilités — validation')
plot_probability_distribution(test_scores, title='Distribution des probabilités — test')

print('Classification report validation')
display(pd.DataFrame(results['validation_report'].classification_report).T)
print('Classification report test')
display(pd.DataFrame(results['test_report'].classification_report).T)

display(results['errors_test'].head(20))

manual_examples = [
    'Entraîner un réseau neuronal convolutif',
    'Construire un pipeline RAG',
    'Déployer un modèle de machine learning',
    'Gérer la relation client',
    'Préparer une réunion commerciale',
    'Utiliser Excel pour suivre un budget',
    '',
    'IA',
    'é',
    'zqxwplm nrvq tppp',
    'Cette compétence vise à automatiser la classification de documents et la génération de réponses avec un modèle de langage dans un contexte métier très concret.'
]
manual_results = []
for example in manual_examples:
    prediction = predict_competence_textcnn(example)
    manual_results.append({'modèle': 'TextCNN', **prediction})
display(pd.DataFrame(manual_results))



# Emplacement des artefacts

Les artefacts et métriques sont enregistrés dans : `artifacts/classification_ia_non_ia_textcnn/`

Le split commun est conservé dans : `artifacts/classification_ia_non_ia_common/splits.csv`



Les artefacts et métriques sont enregistrés dans : `artifacts/classification_ia_non_ia_textcnn/`

Le split commun est conservé dans : `artifacts/classification_ia_non_ia_common/splits.csv`
